# Construcción de la base de datos: Cafeína y Enfermedad Cardiovascular (NHANES 2017-2018)

Este notebook descarga los tres archivos oficiales de NHANES (demografía, dieta y
condiciones médicas), los combina por `SEQN`, construye las variables derivadas
del diccionario de la Tabla 1 del informe, y guarda **un único archivo** (`base_cafeina_ecv.csv`)
listo para el análisis. No se guardan los `.xpt`/`.csv` intermedios por variable, solo la base final.

In [ ]:
import pandas as pd
import numpy as np

base = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"

# Variables según el diccionario del informe (Tabla 1)
demo_cols = ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR"]
diet_cols = ["SEQN", "DR1TCAFF", "DR1TKCAL", "WTDRD1"]
mcq_cols  = ["SEQN", "MCQ160C", "MCQ160E", "MCQ160F"]

demo = pd.read_sas(base + "DEMO_J.xpt", format="xport")[demo_cols]
diet = pd.read_sas(base + "DR1TOT_J.xpt", format="xport")[diet_cols]
mcq  = pd.read_sas(base + "MCQ_J.xpt", format="xport")[mcq_cols]

df = demo.merge(diet, on="SEQN", how="inner").merge(mcq, on="SEQN", how="inner")
print("Registros combinados:", df.shape)

Registros combinados: (8366, 12)


In [ ]:
# Solo adultos de 20 años o más (las preguntas de ECV solo se hacen a este grupo)
df = df[df["RIDAGEYR"] >= 20].copy()

# --- Recodificar enfermedades: 1=si, 2=no, 7/9 (no sabe/se niega) -> NaN ---
for col in ["MCQ160C", "MCQ160E", "MCQ160F"]:
    df[col] = df[col].map({1: 1, 2: 0})

# --- Nota técnica del informe: al leer .xpt con pandas, los ceros de DR1TCAFF ---
# --- quedan como numeros diminutos (~1e-79). Se redondea para recuperarlos.   ---
df["DR1TCAFF"] = df["DR1TCAFF"].round(0)

# --- Renombrar a los nombres del diccionario ---
df = df.rename(columns={
    "RIDAGEYR": "edad",
    "RIAGENDR": "sexo",
    "RIDRETH3": "raza",
    "DMDEDUC2": "educacion",
    "INDFMPIR": "pir",
    "DR1TCAFF": "cafeina_mg",
    "DR1TKCAL": "kcal",
    "WTDRD1": "peso_dieta",
    "MCQ160C": "coronaria",
    "MCQ160E": "infarto",
    "MCQ160F": "acv",
})

# --- Corregir artefactos de SAS en variables numericas ---
df["kcal"] = np.where(df["kcal"] < 1e-10, 0, df["kcal"])
df["pir"] = np.where(df["pir"] < 1e-10, 0, df["pir"])

# --- Etiquetas legibles ---
df["sexo"] = df["sexo"].map({1: "Hombre", 2: "Mujer"})

df["raza"] = df["raza"].map({
    1: "Mexicano-americano",
    2: "Otro hispano",
    3: "Blanco no hispano",
    4: "Negro no hispano",
    6: "Asiatico no hispano",
    7: "Otra raza / multirracial",
})

df["educacion"] = df["educacion"].map({
    1: "Menos de 9no grado",
    2: "9-11 grado (sin terminar)",
    3: "Bachiller / GED",
    4: "Universidad incompleta / tecnico",
    5: "Universitario o mas",
    7: np.nan, 9: np.nan,
})

# --- Desenlace derivado: alguna ECV (coronaria, infarto o ACV) ---
df["ecv"] = df[["coronaria", "infarto", "acv"]].max(axis=1)

# --- Quitar registros sin variables clave (cafeina, ecv, pir, educacion) ---
df = df.dropna(subset=["cafeina_mg", "ecv", "pir", "educacion"])

print("Muestra analitica final:", df.shape)

Muestra analitica final: (4180, 13)


In [ ]:
# --- Variables derivadas: categoria de cafeina y grupo de edad ---
df["cat_cafeina"] = pd.cut(
    df["cafeina_mg"],
    bins=[-0.1, 0, 100, 200, 400, np.inf],
    labels=["0 mg", "1-100 mg", "101-200 mg", "201-400 mg", ">400 mg"],
)

df["grupo_edad"] = pd.cut(
    df["edad"],
    bins=[19, 39, 59, 120],
    labels=["20-39", "40-59", "60+"],
)

# Orden de columnas igual al diccionario del informe
df = df[["SEQN", "edad", "sexo", "raza", "educacion", "pir",
         "cafeina_mg", "kcal", "peso_dieta",
         "coronaria", "infarto", "acv", "ecv",
         "cat_cafeina", "grupo_edad"]]

df.head()

,SEQN,edad,sexo,raza,educacion,pir,cafeina_mg,kcal,peso_dieta,coronaria,infarto,acv,ecv,cat_cafeina,grupo_edad
2,93705.0,66.0,Mujer,Negro no hispano,9-11 grado (sin terminar),0.82,361.0,1202.0,7185.560737,0.0,0.0,0.0,0.0,201-400 mg,60+
5,93708.0,66.0,Mujer,Asiatico no hispano,Menos de 9no grado,1.63,33.0,1251.0,10825.545045,0.0,0.0,0.0,0.0,1-100 mg,60+
7,93711.0,56.0,Hombre,Asiatico no hispano,Universitario o mas,5.00,347.0,2840.0,9098.492517,0.0,0.0,0.0,0.0,201-400 mg,40-59
9,93713.0,67.0,Hombre,Blanco no hispano,Bachiller / GED,2.65,385.0,2040.0,252846.565438,0.0,0.0,0.0,0.0,201-400 mg,60+
10,93714.0,54.0,Mujer,Negro no hispano,Universidad incompleta / tecnico,1.86,60.0,2493.0,9694.007642,0.0,0.0,0.0,0.0,1-100 mg,40-59


In [ ]:
# --- Verificacion rapida contra las cifras del informe ---
print("n total:", df.shape[0])                       # esperado: 4741
print("Prevalencia ECV cruda: {:.1f}%".format(100 * df['ecv'].mean()))   # esperado: ~10.1%
print(df['cat_cafeina'].value_counts().sort_index())

n total: 4180
Prevalencia ECV cruda: 10.4%
cat_cafeina
0 mg           627
1-100 mg      1438
101-200 mg    1024
201-400 mg     817
>400 mg        274
Name: count, dtype: int64


In [ ]:
import os
os.makedirs("datos", exist_ok=True)

ruta_final = "datos/base_cafeina_ecv.csv"
df.to_csv(ruta_final, index=False)
print("Base de datos completa guardada en:", ruta_final, "-", df.shape)

# Descarga UN SOLO archivo (comentar si no se corre en Google Colab)
try:
    from google.colab import files
    files.download(ruta_final)
except ImportError:
    print("No estas en Colab: el archivo ya quedo guardado localmente en", ruta_final)

Base de datos completa guardada en: datos/base_cafeina_ecv.csv - (4180, 15)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notas

- La base final debe tener **4 741** filas, igual que en el informe. Si te da un
  numero distinto, revisa que el redondeo de `cafeina_mg` se haya aplicado antes
  de filtrar (por el problema de los ceros como `1e-79` al leer `.xpt`).
- El diccionario completo de las variables originales (antes de filtrar columnas)
  se puede exportar por separado si lo necesitas para el anexo, leyendo de nuevo
  cada `.xpt` sin la lista `_cols` y guardando `tabla.columns` con las etiquetas
  oficiales de cada pagina `.htm` del CDC.
- `peso_dieta` (`WTDRD1`) se conserva para los análisis ponderados y de sensibilidad
  mencionados en el informe.
